In [2]:
from sklearn.datasets import load_digits
import numpy

In [26]:
data = load_digits()
X = data.data
Y = data.target

In [27]:
# start by applying train/test split to the data
def split_and_normalize(X, Y):
    n = len(X)
    indices = numpy.arange(n)
    numpy.random.shuffle(indices)
    split = int(0.8 * n)
    train_indices = indices[:split]
    test_indices = indices[split:]
    X_train = X[train_indices]
    X_test = X[test_indices]
    Y_train, Y_test = Y[train_indices], Y[test_indices]

    # now normalize the X data based on the mean and standard deviation of the
    # train X data
    means = numpy.mean(X_train, axis = 0)
    stdev = numpy.std(X_train, axis = 0)
    stdev[stdev == 0] = 1
    X_train_norm = (X_train - means)/stdev
    X_test_norm = (X_test - means)/stdev

    # add a column of ones for our intercept
    train_ones = numpy.ones((split, 1))
    test_ones = numpy.ones((n-split, 1))
    X_train_normal = numpy.hstack([train_ones, X_train_norm])
    X_test_normal = numpy.hstack([test_ones, X_test_norm])
    return X_train_normal, X_test_normal, Y_train, Y_test

Next, we code the functions we will use to compute logistic loss, minimize it, and evaluate the model using accuracy. We start with binary classification and later generalize to multiclass regression.

### 1. Logsitc Loss Function
For this we start with the sigmoid function. For a feature vector $x_{i}$, a weight vector $w$, and a label $y_{i}$, we have that the probability that $x_{i}$ gets label $y_{i}$ is
\begin{gather*}
P_{i} = \mathbb{P}(\text{label}(x_{i}) = y_{i}) = \frac{1}{1 + \exp({-y_{i} w \cdot x_{i}})}
\end{gather*}
We multiply $\prod_{i = 1}^{N}P_{i}$ to get the total probability that the model correctly labels all our data. We minimize the loss function
\begin{gather*}
l(w) = -\log\left(\prod_{i = 1}^{N}P_{i}\right) = \sum_{i = 1}^{N}\log\left[{1 + \exp(-y_{i}w\cdot x_{i})}\right]
\end{gather*}
We can add an $\ell_{2}$ regulariztoin term:
\begin{gather*}
l_{r}(w, \lambda) = \sum_{i = 1}^{N}\log\left[{1 + \exp(-y_{i}w\cdot x_{i})}\right] + \lambda||w||^{2}
\end{gather*}

### 2. Gradient 
The gradient of the logistic loss function is
\begin{gather*}
\nabla l(w) = \sum_{i = 1}^{N} \frac{-y_{i} x_{i}}{1 + y_{i} w \cdot x_{i}}
end{gather*}
With regularization this becomes
\begin{gather*}
\nabla l_{r}(w, \lambda) = \sum_{i = 1}^{N} \frac{-y_{i} x_{i}}{1 + \exp{(y_{i} w \cdot x_{i})}} + 2 \lambda w
\end{gather*}

### 3. Minimize the Logistic Loss Function
To minimize the logistic loss function, we iterate
\begin{gather*}
w_{m+1} = w_{m} - \eta \nabla l_{r}(w, \lambda)
\end{gather*}

### 4. Accuracy
To compute the accuracy of our model, we first must find the model's prediction for each data point. To do so, we take the sign of the dot product of the weights vector with the feature vector $x_{i}$:
\begin{gather*}
\text{sign}({w \cdot x_{i}})
\end{gather*}
We then compute whether $w \cdot x_{i} = y_{i}$ using the $\texttt{==}$ boolean operation, which will return an array of ones and zeros. Ones correspond to a data point whose label was correctly predicted, whereas zeroes represent those our model failed to predict. We find the accuracy by taking the number of correct predictions (= the total number of ones) over the total number of entries. We can use $\texttt{numpy.mean()}$ to accomplish this efficiently.

In [32]:
# logistic loss function 
def LogisticLoss(w, X, Y, lam = 0):
    arg = -Y * numpy.dot(X, w)
    unregLoss = numpy.sum(numpy.log(1 + numpy.exp(arg)))
    reg = lam * numpy.dot(w, w)
    return unregLoss + reg

# gradient of the logistic loss function
def logisticGradient(w, X, Y, lam = 0):
    denom = 1 + numpy.exp(Y * numpy.dot(X, w))
    num = -1 * Y * X.transpose()
    frac = num /denom
    unregGrad = numpy.sum(frac, axis = 1)
    gradReg = 2 * lam * w
    gradReg[0] = 0
    return unregGrad + gradReg

# Gradient Descent
def GD(X, Y, iter = 1000, eta = 0.01, lam = 0):
    w = numpy.zeros(X.shape[1])
    for _ in range(iter):
        w = w - eta * logisticGradient(w, X, Y, lam)
    return w

#predictions
def predict(w, X):
    return numpy.sign(numpy.dot(X, w))

def accuracy(w, X, Y):
    return numpy.mean(predict(w, X) == Y)

In [29]:
# extract binary data from our digits data

mask = (Y == 0) | (Y == 1) # array corresponding to the Y array where mask
# [i] is True is Y[i] \in {0, 1} and mask[i] = False otherwise
X_binary = X[mask] # make a new array that is like X except it only has
# the rows where the corresponding value of Y is in {0, 1}
Y_binary = numpy.where(Y[mask] == 0, -1, 1) #goes through each element of 
# Y, and if the element is equal to 0, it gets replaced with -1; otherwise
# gets replaced with 1

In [ ]:
# run binary regression on 0 and 1
X_train_bin, X_test_bin, Y_train_bin, Y_test_bin = split_and_normalize(X_binary, Y_binary)
# learn the weights
wopt = GD(X_train_bin, Y_train_bin)
accTrain = accuracy(wopt, X_train_bin, Y_train_bin)
print("Training Accuracy: ", accTrain)
# test model
accTest = accuracy(wopt, X_test_bin, Y_test_bin)
print("Test accuracy: ", accTest)
print("Generelization error: ", numpy.abs(accTrain - accTest))


Training Accuracy:  1.0
Test accuracy:  1.0
Generelization error:  0.0
__________________________________
Initial loss: 199.62638800126425
Final loss: 0.00928771237896436


In [ ]:
# now try on 3 and 8
mask_prime = (Y == 3) | (Y == 8)
X_binary_prime = X[mask_prime]
Y_binary_prime = numpy.where(Y[mask_prime] == 3, -1, 1)
X_train, X_test, Y_train, Y_test = split_and_normalize(X_binary_prime, Y_binary_prime)
wopt = GD(X_train, Y_train)

print("Initial loss:", LogisticLoss(numpy.zeros(X_train.shape[1]), X_train, Y_train))
print("Final loss:", LogisticLoss(wopt, X_train, Y_train))
print("Training Accuracy:", accuracy(wopt, X_train, Y_train))
print("Test Accuracy:", accuracy(wopt, X_test, Y_test))

Initial loss: 197.54694645958443
Final loss: 0.14336221179368064
Training Accuracy: 1.0
Test Accuracy: 1.0


In [39]:
# confirm with sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

clf = LogisticRegression()
clf.fit(X_train, Y_train)
print("sklearn accuracy:", accuracy_score(Y_test, clf.predict(X_test)))


sklearn accuracy: 1.0


# Multiclass Classification

### Cross Entropy Loss
For the multiclass version of this, we replace our weights vector $w$ with a weigths matrix $W$. We think of $W$ as a stack of column vectors $\vec{w_{j}} \in \mathbb{R}^{d}$, where $d - 1$ is the length of the feature vectors$:

\begin{gather*}
W = 
\begin{bmatrix} 
| & | & & | \\ 
\vec{w_1} & \vec{w_2} & \cdots & \vec{w_k} \\ 
| & | & & |
\end{bmatrix}
\end{gather*}

The probability that a feature vector $\vec{x_{i}}$ receives the label $j$ is given by

\begin{gather*}
\mathbb{P}\{\vec{x_{i}} \text{ gets label }j\} = \text{softmax}(\vec{w_{j}} \cdot \vec{x_{i}})\\
= \frac{\exp{(\vec{w_{j}} \cdot \vec{x_{i}})}}{\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})}
\end{gather*}
Since we assume independence across data points, the probability that our model predicts our results is given by

\begin{gather*}
\prod_{i = 1}^N \mathbb{P}\{\vec{x_{i}} \text{ gets label }y_{i}\} = \prod_{i = 1}^N \frac{\exp{(\vec{w_{y_{i}}} \cdot \vec{x_{i}})}}{\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})}
\end{gather*}

Like before, we minimize the negative logarithm of this function, which is also equal to the logarithm of the reciprocal:
\begin{gather*}
L(W) = \log{\left(\prod_{i = 1}^N \frac{\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})}{\exp{(\vec{w_{y_{i}}} \cdot \vec{x_{i}})}}\right)}\\
= \sum_{i = 1}^N \log\left(\frac{\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})}{\exp{(\vec{w_{y_{i}}} \cdot \vec{x_{i}})}}\right)\\
= \sum_{i = 1}^N \log \left(\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})\right) - \sum_{i = 1}^{N} {(\vec{w_{y_{i}}} \cdot \vec{x_{i}})}
\end{gather*}


### Vectorizing the Cross Entropy Loss

Now that we've calculated the cross entropy loss, we must think about how to implement it in a way that takes advantage of $\texttt{numpy}$'s optimized vectorization capabilities. We start by looking at the first term. Note that simply taking the matrix product $XW$ produces a matrix where the $ij^{\text{th}}$ entry is the dot product $\vec{w_j} \cdot \vec{x_i}$. We can then use $\texttt{numpy}$'s $\texttt{exp(), sum()}$ and $\texttt{log()}$ methods to obtain $\sum_{i = 1}^N \log \left(\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})\right)$.

To cpmute the second term, $\sum_{i = 1}^N (\vec{w_{y_i}} \cdot \vec{x_i})$, we first define the $N \times k $ matrix $Y$ by $Y_{ij}$ equals one if $\vec{x_i}$ has label $j$ and zero otherwise. We take the tranpose $Y^T$ to obtain a matrix that is a horizontal stack of standard basic vectors 

\begin{gather*}
\begin{bmatrix} \vec{e_{y_1}} & \vec{e_{y_2}} & \cdots & \vec{e_{y_N}}
\end{bmatrix}
\end{gather*}

Matrix multiplication is defined such that we can think of the matrix product $W Y^T$ as a horinzontal stack of the product of each of the columns of $Y^T$ with $W$:

\begin{gather*}
W Y^{T} = 
\begin{bmatrix} W\vec{e_{y_1}} & W\vec{e_{y_2}} & \cdots & W\vec{e_{y_N}}
\end{bmatrix}
\end{gather*}

Since the columns of $Y^T$ are standard basis vectors, each picks out the $y_{i}^{\text{th}}$ column of $W$ when multiplied:

\begin{gather*}
W\vec{e_{i}} = \vec{w_i}
\implies WY^T = 
\begin{bmatrix} 
| & | & & | \\ 
\vec{w_{y_1}} & \vec{w_{y_2}} & \cdots & \vec{w_{y_N}} \\ 
| & | & & |
\end{bmatrix}
\end{gather*}

We then repeat our strategy for the first term and take the matrix product of $X$ with this matrix, $XWY^T$. This produces a matrix in which the $ij^{\text{th}}$ entry is equal to $\vec{w_{y_j}}\cdot \vec{x_{i}}$ Since we are only interested when $i = j$, we take the trace of this matrix, and obtain the second term $\sum_{i = 1}^{N} {(\vec{w_{y_{i}}} \cdot \vec{x_{i}})}$.

### Gradient of the Cross Entropy Loss
To minimize the loss function $L(w)$, we must find its gradient. Since the input $W$ is a matrix, its gradient is a matrix. We split up the gradient by term. For the first term we have, by the chain rule and linearity of the gradient,

\begin{gather*}
 \nabla \sum_{i = 1}^N \log \left(\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})\right)\\
  = \sum_{i = 1}^N \nabla \log \left(\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})\right)\\
  = \sum_{i = 1}^N \frac{1}{\left(\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})\right)} \cdot \nabla \left(\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})\right)\\
  = \sum_{i = 1}^N \frac{\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}}) \nabla \vec{w_{m}} \cdot \vec{x_{i}}}{\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})}
\end{gather*}

To find $\nabla \vec{w_{m}} \cdot \vec{x_{i}}$, we rewrite the dot product as the sum $\sum_{c = 1}^d X_{ic}W_{cm}$. This shows that $\frac{\partial \vec{w_{m}} \cdot \vec{x_{i}}}{\partial W_{cm}}$ is equal to a $d \times k$ dimensionsional matrix with $X_{ic}$ in the $cm^{\text{th}}$ slot and zero everywhere else. We get that the gradient is equal to

\begin{gather*}
\sum_{i = 1}^{N}\frac{1}{\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})}\begin{bmatrix}
X_{i1}\exp(\vec{w_1}\cdot\vec{x_i}) & X_{i1}\exp(\vec{w_2}\cdot\vec{x_i}) & \cdots & X_{i1}\exp(\vec{w_k}\cdot\vec{x_i}) \\
X_{i2}\exp(\vec{w_1}\cdot\vec{x_i}) & X_{i2}\exp(\vec{w_2}\cdot\vec{x_i}) & & X_{i2}\exp(\vec{w_k}\cdot\vec{x_i}) \\
\vdots & & \ddots & \vdots \\
X_{id}\exp(\vec{w_1}\cdot\vec{x_i}) & X_{id}\exp(\vec{w_2}\cdot\vec{x_i}) & \cdots & X_{id}\exp(\vec{w_k}\cdot\vec{x_i})
\end{bmatrix}
\end{gather*}

Thus, each feature vector $\vec{x_i}$ has as thr first term in its gradient a $d \times k$ matrix with
\begin{gather*}
\frac{X_{ic}\exp(\vec{w_j}\cdot\vec{x_i})}{\sum_{m = 1}^k \exp (\vec{w_{m}}\cdot \vec{x_{i}})} = X_{ic}\;\text{softmax}({\vec{w_j}\cdot \vec{x_i}})
\end{gather*}
in the $cj^{\text{th}}$ slot.

Next, we must take the gradient of the second term, $\sum_{i = 1}^{N} {(\vec{w_{y_{i}}} \cdot \vec{x_{i}})}$. By linearity of the gradient, this is equal to $\sum_{i = 1}^N\nabla {(\vec{w_{y_{i}}} \cdot \vec{x_{i}})}$ Decomposing the dot product into $\sum_{c = 1}^d X_{ic}W_{cm}$ shows that $\nabla {(\vec{w_{y_{i}}} \cdot \vec{x_{i}})} $ is equal to the $d \times k$ matrix with the feature vector $\vec{x_{i}}$ in the $y_{i}^{\text{th}}$ column and zeroes everywhere else. We thus have that for each data point $\vec{x_{i}}$ with label $y_{i}$, the gradient of the second term of the loss function is the $k \times d$ matrix with entries $X_{ic}\mathbb{I}\{y_{i} = j\}$ in the $cj^{\text{th}}$ slot.

Thus, we have that


\begin{gather*} 
[\nabla L]_{cj} = \frac{\partial L}{\partial W_{cj}} = \sum_{i = 1}^{N} X_{ic} \left( \text{softmax}(\vec{w_{j}} \cdot \vec{x_{i}}) - \mathbb{I}\{y_i = j\}\right)
\end{gather*}

### Vectorization of the Gradient
We begin by defining the $N \times k$ matrix $P$ to be the marix with softmax $(\vec{w_j} \cdot \vec{x_i})$ in its $ij^{\text{th}}$ slot. We call it $P$ because it gives the probability that the feature vector $\vec{x_i}$ gets label $j$. This allows us to rewrite
\begin{gather*}
[\nabla L]_{cj} = \sum_{i = 1}^N X_{ic}(P_{ij} - Y_{ij})
\end{gather*}
We recognize that this almost matches the matrix multiplication formula $AB = \sum_{k = 1}^n A_{ik}B_{kj}$. To rewrite it as a matrix product, we instead use the ${ci}^{\text{th}}$ entry of $X$ transpose:
\begin{gather*}
[\nabla L]_{cj} = \sum_{i = 1}^N [X^T]_{ci}(P_{ij} - Y_{ij})\\
\implies \nabla L = X^T(P - Y)
\end{gather*}
We check that the dimension of the matrix $X^T(P - Y)$ is $d \times k$, as expected.

### Matrix Regularization

The final piece we need to implement multiclass regression is the regularizer. To regularize the matrix, we use the Frobenius norm, which in the sum of all the entries of a matrix sqaured:

\begin{gather*}
|W|_{\text{Frobenius}} = \sum_{i}\sum_{j} W_{ij}^2
\end{gather*}
This gives 
\begin{gather*}
\nabla |W|_{\text{Frobenius}} = 2W
\end{gather*}

which nicely corresponds to the gradient of the $\ell _{2}$ norm of the parameter vector.

In [ ]:
# generate the Y matrix
def label_matrix(length_datatset, num_classes, Y):
    Y_matrix = numpy.zeros((length_datatset, num_classes))
    Y_matrix[numpy.arange(length_datatset), Y] = 1
    return Y_matrix
   
# generate the softmax matrix P 
def softmax_matrix(W, X):
    dot_product_matrix = numpy.dot(X, W) # get w_j \cdot x_i
    unreg_matrix = numpy.exp(dot_product_matrix)
    # normalize by summing across columns
    row_sums = unreg_matrix.sum(axis = 1, keepdims = True)
    return unreg_matrix/row_sums

# calculate the cross entropy loss
def cross_entropy_loss(W, X, Y, num_classes, lam = 0):
    # first calculate the first term of the loss
    dot_product_matrix = numpy.dot(X, W) # get w_j \cdot x_i
    unreg_matrix = numpy.exp(dot_product_matrix)
    row_sums = unreg_matrix.sum(axis = 1)
    first_term = numpy.sum(numpy.log(row_sums))
    # now for the second term
    y_matrix = label_matrix(len(X), num_classes, Y)
    wyi = numpy.dot(W, y_matrix.T)
    wyi_dot_xi = numpy.dot(X, wyi)
    second_term  = numpy.trace(wyi_dot_xi)
    reg = lam * numpy.sum(W**2) # Frobenius norm: sum of all entries squared
    return first_term - second_term + reg

# calculate the gradient of the cross entropy loss using X^T(P - Y) + 2W
def multiclass_gradient(W, X, Y, num_classes, lam = 0):
    loss_gradient = X.T @ (softmax_matrix(W, X) - label_matrix(len(X), num_classes, Y))
    reg_gradient = 2 * lam * W
    return loss_gradient + reg_gradient

# Gradient Descent
def GD(X, Y, num_classes, iter = 1000, eta = 0.01, lam = 0):
    W = numpy.zeros((X.shape[1], num_classes))
    for _ in range(iter):
        W = W - eta * multiclass_gradient(W, X, Y, num_classes, lam)
    return W

# Find the model's prediction of the classification
def predict_multiclass(W, X):
    probabilities = softmax_matrix(W, X) # the softmax matrix tells us 
    # the probability each data point will get label j
    return numpy.argmax(probabilities, axis = 1) # axis = 1 tells us to
    # compare horizontally across columns
    
# Find the accuracy of the model's predictions
def accuracy_multiclass(W, X, Y):
    return numpy.mean(predict_multiclass(W, X) == Y)

    